# **Tech Challenge - Grupo 24: Análise de Retenção de Clientes (Olist)**

Este notebook foca na anÃ¡lise de **Retenção e Fidelidade de Clientes**, utilizando a base consolidada do projeto.

**ResponsÃ¡vel:** Allan Diniz

### **Objetivos:**
*   Calcular Taxa de Recompra e Churn Rate.
*   Identificar fatores que impactam a fidelizaÃ§Ã£o.
*   Propor estratégias de ativaÃ§Ã£o de clientes inativos.

## **1. ImportaÃ§Ã£o de Bibliotecas e ConfiguraÃ§Ãµes**

In [ ]:
try:
    import seaborn as sns
except ImportError:
    !pip install seaborn
    import seaborn as sns

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings('ignore')

# Configurações Visuais
plt.rcParams.update({"figure.facecolor": "#FFFFFF", "axes.facecolor": "#F8F8F8", "font.family": "sans-serif"})
AZUL, LARANJA, VERDE, CINZA = "#2c3e50", "#ff7f0e", "#27ae60", "#B4B2A9"

## **2. Carregamento dos Dados Originais**

Buscando as bases com os dados na nova pasta Datasets.

In [ ]:
# Atualizado para buscar os dados corretamente na pasta Datasets
import os
import pandas as pd

possiveis_caminhos = [
    'Datasets/',
    '1 - Projeto-Olist-KPIs/notebooks/Datasets/',
    '../Datasets/',
    './Datasets/'
]

path_prefix = None
for p in possiveis_caminhos:
    if os.path.exists(p) and os.path.isdir(p):
        path_prefix = p
        break

if path_prefix is None:
    path_prefix = 'Datasets/'

print(f"📂 Folder: {path_prefix}")

try:
    import pandas as pd # Extra assurance
    df_clientes = pd.read_csv(os.path.join(path_prefix, 'olist_customers_dataset.csv'))
    df_pedidos = pd.read_csv(os.path.join(path_prefix, 'olist_orders_dataset.csv'))
    df_itens_pedido = pd.read_csv(os.path.join(path_prefix, 'olist_order_items_dataset.csv'))
    df_pagamentos = pd.read_csv(os.path.join(path_prefix, 'olist_order_payments_dataset.csv'))
    df_avaliacoes = pd.read_csv(os.path.join(path_prefix, 'olist_order_reviews_dataset.csv'))
    df_produtos = pd.read_csv(os.path.join(path_prefix, 'olist_products_dataset.csv'))
    df_vendedores = pd.read_csv(os.path.join(path_prefix, 'olist_sellers_dataset.csv'))
    df_traducao_categorias = pd.read_csv(os.path.join(path_prefix, 'product_category_name_translation.csv'))
    print("✅ Todas as bases foram carregadas com sucesso!")
except Exception as e:
    import pandas as pd # Try one more time
    print(f"❌ Erro ao carregar as bases: {e}")


## **3. Tratamento de Dados (Conforme Grupo)**

In [ ]:
# 1. ConversÃ£o de Datas
colunas_datas = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in colunas_datas: df_pedidos[col] = pd.to_datetime(df_pedidos[col], errors='coerce')

# 2. PadronizaÃ§Ã£o de IDs para o Merge
for df in [df_pedidos, df_itens_pedido, df_pagamentos, df_avaliacoes]:
    if 'order_id' in df.columns: df['order_id'] = df['order_id'].astype(str)
    
# 3. Limpeza de Clientes
df_clientes['customer_zip_code_prefix'] = df_clientes['customer_zip_code_prefix'].astype(str).str.zfill(5)
df_clientes['customer_city'] = df_clientes['customer_city'].str.title()
print("âœ… Tratamento concluÃ­do.")

## **4. Criação da Base Consolidada Oficial**

In [ ]:
# Agrupando Itens (PreÃ§o e Frete total)
df_itens_agrupado = df_itens_pedido.groupby('order_id').agg({'price': 'sum', 'freight_value': 'sum', 'product_id': 'first', 'seller_id': 'first'}).reset_index()

# Agrupando Pagamentos
df_pagamentos_agrupado = df_pagamentos.groupby('order_id').agg({'payment_value': 'sum', 'payment_type': lambda x: '/'.join(x.unique().astype(str)), 'payment_installments': 'max'}).reset_index()

# Agrupando AvaliaÃ§Ãµes
df_avaliacoes_agrupado = df_avaliacoes.groupby('order_id').agg({'review_score': 'mean', 'review_id': 'first'}).reset_index()

# O MERGE CENTRAL
df_consolidado = pd.merge(df_pedidos, df_itens_agrupado, on='order_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_pagamentos_agrupado, on='order_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_avaliacoes_agrupado, on='order_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_produtos, on='product_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_clientes, on='customer_id', how='left')
df_consolidado = pd.merge(df_consolidado, df_vendedores, on='seller_id', how='left')

df_consolidado["atrasado"] = df_consolidado["order_delivered_customer_date"] > df_consolidado["order_estimated_delivery_date"]
print(f"âœ… df_consolidado criada com {df_consolidado.shape[0]} linhas.")

## **5. Análises de Retenção (Allan)**

In [ ]:
# 1. Distribuição de Clientes por Fidelidade
pedidos_por_cliente = df_consolidado.groupby('customer_unique_id')['order_id'].nunique().reset_index(name='qtd_pedidos')
pedidos_por_cliente['categoria'] = pedidos_por_cliente['qtd_pedidos'].apply(lambda x: 'Fiel (>1 compra)' if x > 1 else 'Novo (1 compra)')

plt.figure(figsize=(10, 6))
total = len(pedidos_por_cliente)
ax = sns.countplot(data=pedidos_por_cliente, x='categoria', palette=['#34495e', '#27ae60'], order=['Novo (1 compra)', 'Fiel (>1 compra)'])

plt.title('Retenção: Composição da Base de Clientes', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Status do Cliente', fontsize=12)
plt.ylabel('Quantidade de Clientes', fontsize=12)

# Adicionando % e valores absolutos
for p in ax.patches:
    pct = '{:.1f}%'.format(100 * p.get_height() / total)
    ax.annotate(f'{pct}\n({int(p.get_height()):,})', (p.get_x() + p.get_width() / 2., p.get_height() + 500), 
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='#2d3436')

plt.ylim(0, total * 1.1)
plt.show()

In [ ]:
# 2. Análise de Inatividade (Churn)
# Identifica quanto tempo faz desde a última compra de cada cliente
ultima_compra = df_consolidado.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index()
data_max = df_consolidado['order_purchase_timestamp'].max()
ultima_compra['meses_desde_compra'] = (data_max - ultima_compra['order_purchase_timestamp']).dt.days / 30

plt.figure(figsize=(12, 6))
sns.histplot(ultima_compra['meses_desde_compra'], bins=25, color='#e67e22', kde=True, alpha=0.6)
plt.axvline(x=6, color='#c0392b', linestyle='--', linewidth=2, label='Risco de Churn (6+ meses)')

plt.title('Cronômetro de Inatividade: Quando os clientes somem?', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Meses desde a última compra', fontsize=12)
plt.ylabel('Volume de Clientes', fontsize=12)
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)

# Insight Visual
plt.fill_between([6, ultima_compra['meses_desde_compra'].max()], 0, plt.ylim()[1], color='#c0392b', alpha=0.1, label='Zona de Risco')
plt.show()

## **6. Fatores de Recompra**

In [ ]:
# 3. Impacto da Experiência e Categorias na Fidelização
df_primeira = df_consolidado.sort_values('order_purchase_timestamp').groupby('customer_unique_id').first().reset_index()
df_primeira['fiel'] = df_primeira['customer_unique_id'].isin(pedidos_por_cliente[pedidos_por_cliente['qtd_pedidos'] > 1]['customer_unique_id'])
df_primeira['tipo'] = df_primeira['fiel'].map({True: 'Recorrente', False: 'Única Vez'})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Gráfico A: Score de Satisfação
sns.boxplot(data=df_primeira, x='tipo', y='review_score', ax=ax1, palette=['#ecf0f1', '#3498db'])
ax1.set_title('Impacto do Review Score\n(Clientes que voltam avaliam melhor?)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Tipo de Cliente')
ax1.set_ylabel('Nota (1-5)')

# Gráfico B: Categorias com mais Fidelidade
df_cat = df_consolidado.dropna(subset=['product_category_name'])
df_cat['eh_fiel'] = df_cat['customer_unique_id'].isin(pedidos_por_cliente[pedidos_por_cliente['qtd_pedidos'] > 1]['customer_unique_id'])
top_cats = df_cat.groupby('product_category_name')['eh_fiel'].mean().sort_values(ascending=False).head(10) * 100

top_cats.plot(kind='barh', ax=ax2, color='#27ae60')
ax2.set_title('Top 10 Categorias com Maior Taxa de Recompra', fontsize=14, fontweight='bold')
ax2.set_xlabel('% de Clientes Retidos')
ax2.set_ylabel('Categoria')
ax2.invert_yaxis()

plt.suptitle('Decifrando a Fidelidade: Por que e onde os clientes voltam?', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## **7. Sugestões de Melhoria**

Para um retorno efetivo, propomos trabalhar em **duas frentes** (curto e longo prazo):

1.  **Programa de Pontos**: Criar sistema de recompensa para converter a 2ª compra.
2.  **Lembrete de 'Sumiço'**: Automação para clientes inativos há mais de 4 meses.
3.  **Cuidado com Atrasos**: Cupom de desculpas imediato para problemas de entrega.
4.  **Sugestões do seu Jeito**: Cross-selling baseando na primeira compra.
5.  **Ajuste da Operação (Problema Raiz)**: Negociar novos contratos com transportadoras. Como a logística não acompanhou as vendas, essa parte leva mais tempo (meses de trabalho), mas é a solução real a longo prazo.